In [0]:
%pip install scikit-learn -q

In [0]:
# Optical Degradation Anomaly Detection — Provable Resilience Intelligence
# @ml-engineer | P9-OPTICAL-ANOMALY | Lakelink Fiber
# Detects OSNR degradation and BER exceedances using statistical + Isolation Forest

import mlflow
from pyspark.sql import functions as F
from pyspark.sql.window import Window
import pandas as pd
import numpy as np
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler

SCHEMA = "cdm_tmforum.provable_resilience_intelligence"
EXP = "/Users/stephen.hage@databricks.com/provable-resilience-intelligence/ml_optical_anomaly_detection"
mlflow.set_experiment(EXP)

# Load performance data with windowed features
perf = spark.table(f"{SCHEMA}.resource_performance_measure")

# Per-span rolling statistics (24hr = 96 measurements at 15-min intervals)
w24 = Window.partitionBy("span_id").orderBy("measure_time").rowsBetween(-96, 0)
w_all = Window.partitionBy("span_id")

perf_enriched = perf.select(
    "measure_id", "span_id", "measure_time",
    "osnr_db", "ber", "optical_power_dbm", "chromatic_dispersion", "pmd_ps"
).withColumn("osnr_rolling_avg", F.avg("osnr_db").over(w24)) \
  .withColumn("osnr_rolling_std", F.stddev("osnr_db").over(w24)) \
  .withColumn("ber_rolling_avg", F.avg("ber").over(w24)) \
  .withColumn("ber_rolling_std", F.stddev("ber").over(w24)) \
  .withColumn("power_rolling_avg", F.avg("optical_power_dbm").over(w24)) \
  .withColumn("osnr_global_avg", F.avg("osnr_db").over(w_all)) \
  .withColumn("osnr_global_std", F.stddev("osnr_db").over(w_all)) \
  .withColumn("ber_global_avg", F.avg("ber").over(w_all)) \
  .withColumn("ber_global_std", F.stddev("ber").over(w_all))

# Z-score anomaly flags (statistical baseline)
perf_scored = perf_enriched.withColumn("osnr_zscore",
    F.when(F.col("osnr_global_std") > 0,
           (F.col("osnr_db") - F.col("osnr_global_avg")) / F.col("osnr_global_std")
    ).otherwise(0.0)
).withColumn("ber_zscore",
    F.when(F.col("ber_global_std") > 0,
           (F.col("ber") - F.col("ber_global_avg")) / F.col("ber_global_std")
    ).otherwise(0.0)
).withColumn("osnr_zscore_anomaly",
    F.when(F.col("osnr_zscore") < -3, True).otherwise(False)  # OSNR < 3 sigma below mean
).withColumn("ber_zscore_anomaly",
    F.when(F.col("ber_zscore") > 3, True).otherwise(False)    # BER > 3 sigma above mean
).withColumn("power_drop_anomaly",
    F.when(F.col("optical_power_dbm") < F.col("power_rolling_avg") - 3, True).otherwise(False)
)

print("Z-score anomaly summary (5M rows):")
perf_scored.select(
    F.count("*").alias("total"),
    F.sum(F.col("osnr_zscore_anomaly").cast("int")).alias("osnr_anomalies"),
    F.sum(F.col("ber_zscore_anomaly").cast("int")).alias("ber_anomalies"),
    F.sum(F.col("power_drop_anomaly").cast("int")).alias("power_anomalies")
).show()

In [0]:
# ============================================================
# ISOLATION FOREST on per-span aggregated optical features
# ============================================================

# Build per-span optical feature matrix for IF
span_optical = perf_scored.groupBy("span_id").agg(
    F.avg("osnr_db").alias("avg_osnr"),
    F.stddev("osnr_db").alias("std_osnr"),
    F.min("osnr_db").alias("min_osnr"),
    F.percentile_approx("osnr_db", 0.05).alias("p5_osnr"),
    F.avg("ber").alias("avg_ber"),
    F.max("ber").alias("max_ber"),
    F.stddev("ber").alias("std_ber"),
    F.percentile_approx("ber", 0.95).alias("p95_ber"),
    F.avg("optical_power_dbm").alias("avg_power"),
    F.stddev("optical_power_dbm").alias("std_power"),
    F.avg("chromatic_dispersion").alias("avg_cd"),
    F.avg("pmd_ps").alias("avg_pmd"),
    F.sum(F.col("osnr_zscore_anomaly").cast("int")).alias("osnr_anomaly_count"),
    F.sum(F.col("ber_zscore_anomaly").cast("int")).alias("ber_anomaly_count"),
    F.sum(F.col("power_drop_anomaly").cast("int")).alias("power_anomaly_count"),
    F.count("*").alias("total_measurements")
)

span_optical = span_optical.withColumn("osnr_anomaly_rate", 
    F.col("osnr_anomaly_count") / F.col("total_measurements")
).withColumn("ber_anomaly_rate",
    F.col("ber_anomaly_count") / F.col("total_measurements")
)

IF_FEATURES = ["avg_osnr", "std_osnr", "min_osnr", "p5_osnr",
               "avg_ber", "max_ber", "std_ber", "p95_ber",
               "avg_power", "std_power", "avg_cd", "avg_pmd",
               "osnr_anomaly_rate", "ber_anomaly_rate"]

pdf_if = span_optical.select(["span_id"] + IF_FEATURES + ["osnr_anomaly_count", "ber_anomaly_count", "power_anomaly_count", "total_measurements"]).toPandas().fillna(0)

X_if = pdf_if[IF_FEATURES].values
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_if)

# Train Isolation Forest with MLflow
with mlflow.start_run(run_name="optical_anomaly_iforest") as run:
    contamination = 0.05  # expect ~5% degraded spans
    iforest = IsolationForest(
        n_estimators=200,
        contamination=contamination,
        max_features=0.8,
        random_state=42
    )
    iforest.fit(X_scaled)
    
    scores = iforest.decision_function(X_scaled)
    preds = iforest.predict(X_scaled)  # 1=normal, -1=anomaly
    
    n_anomalies = (preds == -1).sum()
    anomaly_pct = n_anomalies / len(preds) * 100
    
    mlflow.log_param("model_type", "isolation_forest")
    mlflow.log_param("n_estimators", 200)
    mlflow.log_param("contamination", contamination)
    mlflow.log_param("n_features", len(IF_FEATURES))
    mlflow.log_param("n_spans", len(pdf_if))
    mlflow.log_metric("n_anomalies", n_anomalies)
    mlflow.log_metric("anomaly_pct", anomaly_pct)
    mlflow.log_metric("avg_anomaly_score", float(scores.mean()))
    mlflow.log_metric("min_anomaly_score", float(scores.min()))
    
    mlflow.sklearn.log_model(iforest, "model",
        signature=mlflow.models.infer_signature(
            pd.DataFrame(X_scaled, columns=IF_FEATURES),
            pd.DataFrame({'anomaly_score': scores, 'is_anomaly': (preds == -1).astype(int)})
        ))
    mlflow.set_tag("project", "provable_resilience_intelligence")
    mlflow.set_tag("brand", "lakelink_fiber")
    mlflow.set_tag("task_code", "P9-OPTICAL-ANOMALY")
    if_run_id = run.info.run_id

pdf_if['anomaly_score'] = scores
pdf_if['is_anomaly'] = (preds == -1).astype(int)
pdf_if['anomaly_severity'] = pd.cut(scores, bins=[-np.inf, -0.15, -0.05, np.inf],
                                     labels=['severe', 'moderate', 'normal'])

print(f"Isolation Forest results:")
print(f"  Total spans: {len(pdf_if)}")
print(f"  Anomalies: {n_anomalies} ({anomaly_pct:.1f}%)")
print(f"  Score range: [{scores.min():.3f}, {scores.max():.3f}]")
print(f"\nAnomaly severity breakdown:")
print(pdf_if['anomaly_severity'].value_counts().to_string())
print(f"\nMLflow run: {if_run_id}")

In [0]:
# Write anomaly detection results
output_cols = ['span_id', 'avg_osnr', 'min_osnr', 'p5_osnr', 'avg_ber', 'max_ber', 'p95_ber',
               'avg_power', 'std_power', 'avg_cd', 'avg_pmd',
               'osnr_anomaly_count', 'ber_anomaly_count', 'power_anomaly_count',
               'total_measurements', 'anomaly_score', 'is_anomaly', 'anomaly_severity']

anom_df = spark.createDataFrame(pdf_if[output_cols])
anom_df = anom_df.withColumn("detected_at", F.current_timestamp()) \
    .withColumn("model_run_id", F.lit(if_run_id))

anom_df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{SCHEMA}.optical_anomaly_detection")

print(f"Wrote {anom_df.count()} rows to {SCHEMA}.optical_anomaly_detection")
anom_df.filter("is_anomaly = 1").select("span_id", "anomaly_score", "anomaly_severity",
    "osnr_anomaly_count", "ber_anomaly_count", "avg_osnr", "min_osnr").orderBy("anomaly_score").show(10)

spark.sql(f"""COMMENT ON TABLE {SCHEMA}.optical_anomaly_detection IS
  'Optical degradation anomaly detection results. Isolation Forest on 14 OSNR/BER/power features per span. Z-score + IF dual detection. anomaly_severity: severe/moderate/normal. Lakelink Fiber Provable Resilience Intelligence.'""")
print("Done.")